# Methodology Brain Figure: `shapesphysical`

This notebook is a focused, thesis-oriented adaptation of the brain EDA in notebooks `04` and `05`.

It uses one retained `shapesphysical` run and produces three clearer outputs:

- preprocessing and proxy ROI overlays
- transcript-to-brain alignment views
- Schaefer overlap and parcel correlation summaries

These figures are intended to support the methodology chapter and the appendix, not the main results chapter.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import nibabel as nib
from nibabel.processing import resample_from_to
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 80)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'eda_brain_data').exists() and (candidate / 'thesis_methodology').exists():
            return candidate
    raise FileNotFoundError('Could not locate the thesis_neuro project root from the current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd())
SUBJECT = 'sub-243'
TASK = 'shapesphysical'

RAW_FUNC_PATH = PROJECT_ROOT / f'eda_brain_data/datasets/ds002345_shapesphysical_subset/{SUBJECT}/func/{SUBJECT}_task-{TASK}_bold.nii.gz'
PREPROC_MATCHES = sorted(
    (PROJECT_ROOT / 'eda_brain_data/datasets/ds002345/derivatives').glob(
        f'fmriprep_shapesphysical_batches/batch_*/out/{SUBJECT}/func/{SUBJECT}_task-{TASK}_space-MNI152NLin6Asym_desc-preproc_bold.nii.gz'
    )
)
if not PREPROC_MATCHES:
    raise FileNotFoundError(f'No desc-preproc BOLD file found for {SUBJECT} {TASK}.')
PREPROC_FUNC_PATH = PREPROC_MATCHES[0]

ATLAS_PATH = PROJECT_ROOT / 'eda_brain_data/assets/roi/Schaefer2018_200Parcels_7Networks_order_FSLMNI152_1mm.nii.gz'
TRANSCRIPT_PATH = PROJECT_ROOT / f'eda_brain_data/datasets/data/transcripts/{TASK}/{TASK}_tr_aligned.tsv'
PHONEME_PATH = PROJECT_ROOT / f'eda_brain_data/datasets/data/transcripts/{TASK}/{TASK}_phonemes_by_tr.tsv'
OUTPUT_DIR = PROJECT_ROOT / 'thesis_methodology/figures/methodology'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_SPECS = [
    {'roi': 'left_auditory_proxy', 'label': 'Left auditory', 'bounds': ((0.10, 0.30), (0.48, 0.68), (0.18, 0.42))},
    {'roi': 'right_auditory_proxy', 'label': 'Right auditory', 'bounds': ((0.70, 0.90), (0.48, 0.68), (0.18, 0.42))},
    {'roi': 'left_tpj_proxy', 'label': 'Left TPJ', 'bounds': ((0.08, 0.28), (0.32, 0.54), (0.34, 0.62))},
    {'roi': 'right_tpj_proxy', 'label': 'Right TPJ', 'bounds': ((0.72, 0.92), (0.32, 0.54), (0.34, 0.62))},
    {'roi': 'posterior_cingulate_proxy', 'label': 'PCC / precuneus', 'bounds': ((0.40, 0.60), (0.28, 0.50), (0.40, 0.76))},
    {'roi': 'medial_prefrontal_proxy', 'label': 'mPFC', 'bounds': ((0.42, 0.58), (0.58, 0.86), (0.38, 0.72))},
]

ROI_COLORS = {
    'whole_brain': '#111111',
    'left_auditory_proxy': '#1f77b4',
    'right_auditory_proxy': '#5aa3d6',
    'left_tpj_proxy': '#2ca02c',
    'right_tpj_proxy': '#7bc87c',
    'posterior_cingulate_proxy': '#ff7f0e',
    'medial_prefrontal_proxy': '#d62728',
}

print('RAW_FUNC_PATH:', RAW_FUNC_PATH)
print('PREPROC_FUNC_PATH:', PREPROC_FUNC_PATH)
print('ATLAS_PATH:', ATLAS_PATH)
print('TRANSCRIPT_PATH:', TRANSCRIPT_PATH)


In [ ]:
def robust_brain_mask(mean_img: np.ndarray) -> np.ndarray:
    positive = mean_img[mean_img > 0]
    if positive.size == 0:
        raise ValueError('Mean image is empty; could not create a brain mask.')
    threshold = np.percentile(positive, 40)
    mask = mean_img > threshold
    mask = ndi.binary_closing(mask, iterations=2)
    mask = ndi.binary_opening(mask, iterations=1)
    labels, n_labels = ndi.label(mask)
    if n_labels == 0:
        return mask
    counts = np.bincount(labels.ravel())
    counts[0] = 0
    return labels == counts.argmax()


def mask_bounds(mask: np.ndarray):
    coords = np.argwhere(mask)
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0) + 1
    return mins, maxs


def fractional_box_mask(brain_mask: np.ndarray, fractions) -> np.ndarray:
    mins, maxs = mask_bounds(brain_mask)
    slices = []
    for axis, (low_frac, high_frac) in enumerate(fractions):
        start = int(round(mins[axis] + (maxs[axis] - mins[axis]) * low_frac))
        stop = int(round(mins[axis] + (maxs[axis] - mins[axis]) * high_frac))
        start = max(start, int(mins[axis]))
        stop = min(max(stop, start + 1), int(maxs[axis]))
        slices.append(slice(start, stop))
    out = np.zeros_like(brain_mask, dtype=bool)
    out[slices[0], slices[1], slices[2]] = True
    return out & brain_mask


def build_proxy_rois(brain_mask: np.ndarray) -> dict:
    return {spec['roi']: fractional_box_mask(brain_mask, spec['bounds']) for spec in ROI_SPECS}


def centered_slices(mask: np.ndarray):
    coords = np.argwhere(mask)
    center = np.round(coords.mean(axis=0)).astype(int)
    return tuple(int(v) for v in center)


def overlay_slice(ax, base_slice, overlays, title: str):
    ax.imshow(np.rot90(base_slice), cmap='gray')
    for overlay_mask, color, alpha in overlays:
        if overlay_mask.any():
            masked = np.ma.masked_where(~overlay_mask, overlay_mask)
            ax.imshow(np.rot90(masked), cmap=ListedColormap([color]), interpolation='nearest', alpha=alpha)
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()


def zscore(values: pd.Series) -> pd.Series:
    std = values.std()
    if std == 0 or np.isnan(std):
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - values.mean()) / std


def load_transcript_features() -> pd.DataFrame:
    tr_df = pd.read_csv(TRANSCRIPT_PATH, sep='\t')
    tr_df['text'] = tr_df['text'].fillna('')
    tr_df['word_count'] = tr_df['text'].str.split().map(len)
    if PHONEME_PATH.exists():
        phon_df = pd.read_csv(PHONEME_PATH, sep='\t')[['tr_index', 'phonemes']]
        phon_df['phonemes'] = phon_df['phonemes'].fillna('')
        phon_df['phoneme_count'] = phon_df['phonemes'].str.split().map(len)
        tr_df = tr_df.merge(phon_df[['tr_index', 'phoneme_count']], on='tr_index', how='left')
    else:
        tr_df['phoneme_count'] = np.nan
    return tr_df[['tr_index', 'text', 'word_count', 'phoneme_count']]


def extract_region_timecourses(data_4d: np.ndarray, region_masks: dict) -> pd.DataFrame:
    n_trs = data_4d.shape[-1]
    flat_data = data_4d.reshape(-1, n_trs)
    rows = {'tr_index': np.arange(n_trs)}
    for region_name, mask in region_masks.items():
        flat_mask = mask.reshape(-1)
        rows[region_name] = flat_data[flat_mask].mean(axis=0) if flat_mask.any() else np.full(n_trs, np.nan)
    return pd.DataFrame(rows)


def lagged_correlations(frame: pd.DataFrame, region_names, feature: str, max_lag: int = 6) -> pd.DataFrame:
    rows = []
    for region_name in region_names:
        for lag in range(max_lag + 1):
            shifted = frame[feature].shift(lag)
            valid = frame[region_name].notna() & shifted.notna()
            corr = float(np.corrcoef(frame.loc[valid, region_name], shifted.loc[valid])[0, 1]) if valid.sum() >= 8 else np.nan
            rows.append({'region': region_name, 'feature': feature, 'lag_tr': lag, 'correlation': corr})
    return pd.DataFrame(rows)


def resample_schaefer_to_run(func_img):
    atlas_img = nib.load(str(ATLAS_PATH))
    resampled = resample_from_to(atlas_img, (func_img.shape[:3], func_img.affine), order=0)
    return np.rint(resampled.get_fdata()).astype(int)


def atlas_overlap_diagnostics(atlas_data: np.ndarray, brain_mask: np.ndarray) -> pd.Series:
    atlas_mask = atlas_data > 0
    overlap = atlas_mask & brain_mask
    return pd.Series({
        'atlas_voxels': int(atlas_mask.sum()),
        'brain_mask_voxels': int(brain_mask.sum()),
        'overlap_voxels': int(overlap.sum()),
        'overlap_fraction_of_brain': float(overlap.sum() / max(brain_mask.sum(), 1)),
        'overlap_fraction_of_atlas': float(overlap.sum() / max(atlas_mask.sum(), 1)),
        'observed_parcel_count': int(len(np.unique(atlas_data[atlas_data > 0]))),
    })


def summarize_schaefer_parcels(data_4d: np.ndarray, mean_img: np.ndarray, std_img: np.ndarray, atlas_data: np.ndarray):
    flat_data = data_4d.reshape(-1, data_4d.shape[-1])
    flat_mean = mean_img.reshape(-1)
    flat_std = std_img.reshape(-1)
    flat_atlas = atlas_data.reshape(-1)
    rows = []
    tc = {'tr_index': np.arange(data_4d.shape[-1])}
    for parcel_id in sorted(int(v) for v in np.unique(atlas_data) if v > 0):
        mask = flat_atlas == parcel_id
        if mask.sum() < 10:
            continue
        parcel_key = f'parcel_{parcel_id:03d}'
        tc[parcel_key] = flat_data[mask].mean(axis=0)
        tsnr = np.divide(flat_mean[mask], flat_std[mask], out=np.zeros(mask.sum(), dtype=np.float32), where=flat_std[mask] > 0)
        rows.append({
            'parcel_id': parcel_id,
            'parcel_key': parcel_key,
            'n_voxels': int(mask.sum()),
            'mean_bold': float(flat_mean[mask].mean()),
            'median_tsnr': float(np.median(tsnr)),
        })
    return pd.DataFrame(rows), pd.DataFrame(tc)


def parcel_feature_correlations(parcel_tc_df: pd.DataFrame, transcript_df: pd.DataFrame, feature: str, max_lag: int = 6) -> pd.DataFrame:
    merged = parcel_tc_df.merge(transcript_df[['tr_index', feature]], on='tr_index', how='left')
    rows = []
    for parcel in [c for c in parcel_tc_df.columns if c != 'tr_index']:
        best_corr = np.nan
        best_lag = np.nan
        for lag in range(max_lag + 1):
            shifted = merged[feature].shift(lag)
            valid = merged[parcel].notna() & shifted.notna()
            if valid.sum() < 8:
                continue
            corr = float(np.corrcoef(merged.loc[valid, parcel], shifted.loc[valid])[0, 1])
            if np.isnan(best_corr) or abs(corr) > abs(best_corr):
                best_corr = corr
                best_lag = lag
        rows.append({'parcel_key': parcel, 'feature': feature, 'best_lag_tr': best_lag, 'best_correlation': best_corr})
    out = pd.DataFrame(rows)
    out['parcel_id'] = out['parcel_key'].str.extract(r'(\d+)').astype(float).astype('Int64')
    return out.sort_values('best_correlation', key=lambda s: s.abs(), ascending=False)


In [ ]:
raw_img = nib.load(str(RAW_FUNC_PATH))
preproc_img = nib.load(str(PREPROC_FUNC_PATH))
raw_data = raw_img.get_fdata(dtype=np.float32)
preproc_data = preproc_img.get_fdata(dtype=np.float32)

raw_mean = raw_data.mean(axis=3)
preproc_mean = preproc_data.mean(axis=3)
preproc_std = preproc_data.std(axis=3)
raw_mask = robust_brain_mask(raw_mean)
brain_mask = robust_brain_mask(preproc_mean)
proxy_rois = build_proxy_rois(brain_mask)
atlas_data = resample_schaefer_to_run(preproc_img)
atlas_diag = atlas_overlap_diagnostics(atlas_data, brain_mask)
parcel_summary, parcel_tc = summarize_schaefer_parcels(preproc_data, preproc_mean, preproc_std, atlas_data)
transcript_df = load_transcript_features()
region_tc = extract_region_timecourses(preproc_data, {'whole_brain': brain_mask} | proxy_rois)
merged = region_tc.merge(transcript_df, on='tr_index', how='left')
lag_word = lagged_correlations(merged, ['whole_brain'] + [spec['roi'] for spec in ROI_SPECS], 'word_count')
lag_phoneme = lagged_correlations(merged, ['whole_brain'] + [spec['roi'] for spec in ROI_SPECS], 'phoneme_count')
parcel_word = parcel_feature_correlations(parcel_tc, transcript_df, 'word_count').merge(parcel_summary, on=['parcel_id', 'parcel_key'], how='left')
parcel_phoneme = parcel_feature_correlations(parcel_tc, transcript_df, 'phoneme_count').merge(parcel_summary, on=['parcel_id', 'parcel_key'], how='left')

raw_x_mid, raw_y_mid, raw_z_mid = centered_slices(raw_mask)
x_mid, y_mid, z_mid = centered_slices(brain_mask)

print(atlas_diag)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

overlay_slice(axes[0, 0], raw_mean[raw_x_mid, :, :], [], 'Raw sagittal')
overlay_slice(axes[0, 1], raw_mean[:, raw_y_mid, :], [], 'Raw coronal')
overlay_slice(axes[0, 2], raw_mean[:, :, raw_z_mid], [], 'Raw axial')

overlay_slice(
    axes[1, 0],
    preproc_mean[x_mid, :, :],
    [(mask[x_mid, :, :], ROI_COLORS[name], 0.30) for name, mask in proxy_rois.items()],
    'Post-preproc + proxy ROIs (sagittal)',
)
overlay_slice(
    axes[1, 1],
    preproc_mean[:, y_mid, :],
    [(mask[:, y_mid, :], ROI_COLORS[name], 0.30) for name, mask in proxy_rois.items()],
    'Post-preproc + proxy ROIs (coronal)',
)
overlay_slice(
    axes[1, 2],
    preproc_mean[:, :, z_mid],
    [(mask[:, :, z_mid], ROI_COLORS[name], 0.30) for name, mask in proxy_rois.items()],
    'Post-preproc + proxy ROIs (axial)',
)

handles = [plt.Line2D([0], [0], color=ROI_COLORS[spec['roi']], lw=5, label=spec['label']) for spec in ROI_SPECS]
fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Shapesphysical: before preprocessing and proxy ROI inspection', fontsize=15, y=0.98)
fig.tight_layout(rect=(0, 0.05, 1, 0.95))

out_path = OUTPUT_DIR / '13_brain_eda_shapesphysical_preproc_and_rois.png'
fig.savefig(out_path, dpi=180, bbox_inches='tight')
plt.show()
print('Wrote:', out_path)


In [ ]:
fig = plt.figure(figsize=(13, 8))
gs = fig.add_gridspec(2, 2, width_ratios=[1.25, 1.0], hspace=0.28, wspace=0.24)
ax_tc = fig.add_subplot(gs[0, 0])
ax_transcript = fig.add_subplot(gs[1, 0], sharex=ax_tc)
ax_heat_word = fig.add_subplot(gs[0, 1])
ax_heat_phoneme = fig.add_subplot(gs[1, 1])

ax_tc.plot(merged['tr_index'], zscore(merged['whole_brain']), label='Whole brain', color=ROI_COLORS['whole_brain'], lw=2.2)
for spec in ROI_SPECS:
    ax_tc.plot(merged['tr_index'], zscore(merged[spec['roi']]), label=spec['label'], color=ROI_COLORS[spec['roi']], lw=1.2, alpha=0.9)
ax_tc.set_title('Whole-brain and proxy ROI timecourses')
ax_tc.set_ylabel('z-scored signal')
ax_tc.legend(loc='upper right', ncol=2, fontsize=8)

ax_transcript.bar(merged['tr_index'], merged['word_count'].fillna(0), color='#4c78a8', alpha=0.75, label='Words / TR')
ax_transcript.plot(merged['tr_index'], merged['phoneme_count'].fillna(0), color='#f58518', lw=1.6, label='Phonemes / TR')
ax_transcript.set_title('Transcript density on the same TR grid')
ax_transcript.set_xlabel('TR index')
ax_transcript.set_ylabel('Transcript density')
ax_transcript.legend(loc='upper right')

ordered_regions = ['whole_brain'] + [spec['roi'] for spec in ROI_SPECS]
region_labels = ['Whole brain'] + [spec['label'] for spec in ROI_SPECS]

for ax, lag_df, title in [
    (ax_heat_word, lag_word, 'Lag correlations: word count'),
    (ax_heat_phoneme, lag_phoneme, 'Lag correlations: phoneme count'),
]:
    pivot = lag_df.pivot(index='region', columns='lag_tr', values='correlation').reindex(ordered_regions)
    im = ax.imshow(pivot.values, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(region_labels)))
    ax.set_yticklabels(region_labels)
    ax.set_xlabel('Lag (TRs)')
    ax.set_title(title)

fig.colorbar(im, ax=[ax_heat_word, ax_heat_phoneme], fraction=0.03, pad=0.02, label='Pearson r')
fig.suptitle('Shapesphysical: transcript-to-brain alignment checks', fontsize=15, y=0.98)
fig.tight_layout(rect=(0, 0, 1, 0.96))

out_path = OUTPUT_DIR / '14_brain_eda_shapesphysical_alignment.png'
fig.savefig(out_path, dpi=180, bbox_inches='tight')
plt.show()
print('Wrote:', out_path)


In [ ]:
fig = plt.figure(figsize=(13, 8))
gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 1, 1.2], hspace=0.25, wspace=0.18)
ax_sag = fig.add_subplot(gs[0, 0])
ax_cor = fig.add_subplot(gs[0, 1])
ax_axi = fig.add_subplot(gs[0, 2])
ax_bar = fig.add_subplot(gs[:, 3])
ax_text = fig.add_subplot(gs[1, :3])

atlas_mask = atlas_data > 0
overlap_mask = atlas_mask & brain_mask

overlay_slice(ax_sag, preproc_mean[x_mid, :, :], [(brain_mask[x_mid, :, :], '#b8c6d1', 0.12), (atlas_mask[x_mid, :, :], '#4c78a8', 0.18), (overlap_mask[x_mid, :, :], '#d62728', 0.32)], 'Schaefer overlap (sagittal)')
overlay_slice(ax_cor, preproc_mean[:, y_mid, :], [(brain_mask[:, y_mid, :], '#b8c6d1', 0.12), (atlas_mask[:, y_mid, :], '#4c78a8', 0.18), (overlap_mask[:, y_mid, :], '#d62728', 0.32)], 'Schaefer overlap (coronal)')
overlay_slice(ax_axi, preproc_mean[:, :, z_mid], [(brain_mask[:, :, z_mid], '#b8c6d1', 0.12), (atlas_mask[:, :, z_mid], '#4c78a8', 0.18), (overlap_mask[:, :, z_mid], '#d62728', 0.32)], 'Schaefer overlap (axial)')

top_word = parcel_word.dropna(subset=['best_correlation']).head(12).iloc[::-1]
ax_bar.barh(top_word['parcel_key'], top_word['best_correlation'], color='#4c78a8')
ax_bar.set_title('Top Schaefer parcels\nby word-count correlation')
ax_bar.set_xlabel('Best Pearson r')

ax_text.axis('off')
ax_text.text(0.00, 0.78, f"Atlas voxels: {int(atlas_diag['atlas_voxels']):,}", fontsize=11)
ax_text.text(0.00, 0.58, f"Brain-mask voxels: {int(atlas_diag['brain_mask_voxels']):,}", fontsize=11)
ax_text.text(0.00, 0.38, f"Overlap voxels: {int(atlas_diag['overlap_voxels']):,}", fontsize=11)
ax_text.text(0.38, 0.78, f"Brain overlap: {atlas_diag['overlap_fraction_of_brain']:.2%}", fontsize=11)
ax_text.text(0.38, 0.58, f"Atlas overlap: {atlas_diag['overlap_fraction_of_atlas']:.2%}", fontsize=11)
ax_text.text(0.38, 0.38, f"Observed parcels: {int(atlas_diag['observed_parcel_count'])}", fontsize=11)

fig.suptitle('Shapesphysical: Schaefer overlap and parcel-level transcript alignment', fontsize=15, y=0.98)
fig.tight_layout(rect=(0, 0, 1, 0.96))

out_path = OUTPUT_DIR / '15_brain_eda_shapesphysical_schaefer.png'
fig.savefig(out_path, dpi=180, bbox_inches='tight')
plt.show()
print('Wrote:', out_path)

display(parcel_word[['parcel_key', 'best_lag_tr', 'best_correlation', 'n_voxels', 'median_tsnr']].head(12).round(3))
display(parcel_phoneme[['parcel_key', 'best_lag_tr', 'best_correlation', 'n_voxels', 'median_tsnr']].head(12).round(3))
